In [1]:
import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers import util
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification, BertTokenizer, BertModel
import torch.nn as nn

import torch
import dgl
import os
from dgl import save_graphs, load_graphs
from dgl.data.utils import makedirs, save_info, load_info, save_graphs, load_graphs
from tqdm import tqdm
import json

import dgl.data
import matplotlib.pyplot as plt




/home/adi/Dev/CaseGNN/.venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [2]:

# Load Model
device = torch.device('cuda')
model_name = 'CSHaitao/SAILER_en_finetune'
model = AutoModel.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)



In [3]:
type_feature='fact'
type_dataset='test'

In [4]:

candidate_matrix = torch.load("../PromptCase/promptcase_embedding/"+type_dataset+"_fact_issue_cross_embedding.pt")

with open("../PromptCase/promptcase_embedding/"+type_dataset+"_fact_issue_cross_embedding_case_list.json", "rb")as fIn:
    candidate_matrix_index = json.load(fIn) 


/tmp/ipykernel_9285/4269081105.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  candidate_matrix = torch.load("../PromptCase/promptcase_embedding/"+type_dataset+"_fact_is

In [5]:

if type_feature == 'fact':
    ie_path = "../DATASET/Relations-fact-"+type_dataset+"/result/"
    embedding_index = 0
elif type_feature == 'issue':
    ie_path = "../DATASET/Relations-issue-"+type_dataset+"/result/"
    embedding_index = 1


In [6]:

file_list = os.listdir(ie_path)

graph_num = 0
graph_list = []
graph_labels = {}
graph_name_list = []
zero_file = []

model.eval()


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [7]:

with torch.no_grad():
    for file in tqdm(file_list):
        graph_num += 1
        file_name = file.split('.')[0]
        ## case_embedding_format = [fact_embedding, issue_embedding, cross_embedding]
        promptcase_embedding = candidate_matrix[0][embedding_index][candidate_matrix_index.index(file_name+'.txt')]
        graph_name_list.append(int(file_name.split('_')[-1]))
        list_node1 = []
        list_node2 = []
        list_relation = []
        index_dict = {}
        node_num = -1
        Relation_embedding_weights = []
        node_embedding_weights = []
        split_txt_list = []
                
        with open(ie_path+file, "r") as f:
            relation_triplets = f.readlines()
            for line in relation_triplets:
                if line == 'Type,Entity1,Relationship,Type,Entity2\n':
                    node_num += 1
                    index_dict.update({'promptcase_node': node_num})
                    node_embedding_weights.append(promptcase_embedding)
                    list_node1.append(node_num)
                    list_node2.append(node_num)
                    Relation_embedding_weights.append(promptcase_embedding)
                    list_node2.append(node_num)
                    list_node1.append(node_num)
                    Relation_embedding_weights.append(promptcase_embedding)
                    
                else:
                    if '×' in line:
                        line = line.replace('×','')
                    a_1 = line[:-1].split(',')
                    split_txt = [a_1[1], a_1[2], a_1[4]]
                    if split_txt in split_txt_list:
                        continue
                    else:
                        tokenized = tokenizer(split_txt, return_tensors='pt', padding=True, truncation=True).to(device)
                        embedding = model(**tokenized)
                        cls_embedding = embedding[0][:,0] ##cls token embedding [1,768]
                        cls_embedding = cls_embedding.to('cpu')
                        Entity_1_embedding = cls_embedding[0]
                        Entity_2_embedding = cls_embedding[2]
                        Relation = cls_embedding[1]
                        if a_1[1] in index_dict.keys():
                            Entity_1 = index_dict[a_1[1]]
                        else:
                            node_num += 1
                            Entity_1 = node_num
                            index_dict.update({a_1[1]: Entity_1})
                            node_embedding_weights.append(Entity_1_embedding)
                        if a_1[4] in index_dict.keys():
                            Entity_2 = index_dict[a_1[4]]
                        else:
                            node_num += 1
                            Entity_2 = node_num
                            index_dict.update({a_1[4]: Entity_2})
                            node_embedding_weights.append(Entity_1_embedding)
                        list_node1.append(Entity_1)
                        list_node2.append(Entity_2)
                        Relation_embedding_weights.append(Relation)
                        
                        list_node1.append(Entity_2)
                        list_node2.append(Entity_1)
                        Relation_embedding_weights.append(Relation)
                        
                        list_node1.append(index_dict['promptcase_node'])
                        list_node2.append(Entity_1)
                        Relation_embedding_weights.append(Entity_1_embedding)
                        
                        list_node1.append(Entity_1)
                        list_node2.append(index_dict['promptcase_node'])
                        Relation_embedding_weights.append(Entity_1_embedding)
                        
                        list_node1.append(index_dict['promptcase_node'])
                        list_node2.append(Entity_2)
                        Relation_embedding_weights.append(Entity_2_embedding)
                        
                        list_node1.append(Entity_2)
                        list_node2.append(index_dict['promptcase_node'])
                        Relation_embedding_weights.append(Entity_2_embedding)
                        split_txt_list.append(split_txt)

            f.close()
            print(file)

        # Graph Construction
        g = dgl.graph((list_node1, list_node2))
        g_1 = g

        if len(Relation_embedding_weights) == 0:
            b = 0
            c = 0
            print(file_name, ': zero node and edge')
            zero_file.append(file_name)
        else:
            print('Relation num:', len(Relation_embedding_weights))
            print('Node num:', len(node_embedding_weights))
            b = torch.stack(Relation_embedding_weights)
            c = torch.stack(node_embedding_weights)

            g_1.ndata['w'] = c
            g_1.edata['w'] = b
            graph_list.append(g_1)


  1%|          | 1/133 [00:01<02:16,  1.04s/it]

named_entity_5801.csv
Relation num: 296
Node num: 60


  2%|▏         | 3/133 [00:01<01:01,  2.12it/s]

named_entity_3472.csv
Relation num: 434
Node num: 84
named_entity_4245.csv
Relation num: 164
Node num: 37


  3%|▎         | 4/133 [00:02<01:09,  1.85it/s]

...

named_entity_4193.csv
Relation num: 974
Node num: 176


 99%|█████████▉| 132/133 [01:19<00:00,  1.45it/s]

named_entity_4585.csv
Relation num: 758
Node num: 137


100%|██████████| 133/133 [01:20<00:00,  1.66it/s]

named_entity_995.csv
Relation num: 224
Node num: 48


In [8]:

tensor_graph_name = torch.FloatTensor(graph_name_list)
graph_labels.update({'name_list': tensor_graph_name})

save_graphs("./graph/graph_bin"+"/bidirec_"+type_dataset+"_"+type_feature+".bin", graph_list, graph_labels)
print('Graph saved.')
if len(zero_file) != 0:
    print(zero_file)


Graph saved.


In [9]:

# test/train synthetic graph construction
if type_dataset == 'test':
    graph_label = {"glabel": tensor_graph_name}
    save_graphs("./graph/graph_bin"+"/bidirec_"+type_dataset+"_"+type_feature+"_Synthetic.bin", graph_list, graph_label)
elif type_dataset == 'train':
    CaseGraph = {}
    labels = tensor_graph_name.tolist()
    for i in range(len(labels)):
        CaseGraph[str(int(labels[i])).zfill(6)] = graph_list[i]
    with open('../label/train'+'.json', 'r') as f:
        noticed_case_list = json.load(f)
        f.close()
    query_graph_dict = {}
    query_graph_list = []
    query_graph_label = []
    for key, value in noticed_case_list.items():
        k = key.split('.')[0]
        query_graph_dict.update({k: (CaseGraph[str(k).zfill(6)])})
        query_graph_list.append(CaseGraph[str(k).zfill(6)])
        query_graph_label.append(int(k))
    graph_labels = {"glabel": torch.Tensor(query_graph_label)}

    save_graphs("./graph/graph_bin"+"/bidirec_"+type_dataset+"_"+type_feature+"_Synthetic.bin", query_graph_list, graph_labels)



In [1]:
from dgl import load_graphs

gllis, gl = load_graphs("./graph/graph_bin"+"/bidirec_"+"train"+"_"+"issue"+".bin")

/home/adi/Dev/CaseGNN/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
gl

{'name_list': tensor([ 532., 5291.,  462.,  669., 7044., 5801.,   57.,  473., 3838., 3472.,
         4245., 1862., 3534., 4383., 6465., 1246., 3822.,  250.,  330., 6450.,
         1840., 1662., 4455.,  488.,  328.,  586., 3249., 3650., 2873.,  591.,
         1455., 5920., 1501., 2976.,  223., 1829.,  720.,  554., 4995.,  215.,
         3291.,  343., 1213., 3962.,  524., 3864., 2465.,  579.,  928.,  257.,
         2724., 3289., 4744., 2803.,  204., 3687., 1576., 1214., 2462., 4201.,
         3063., 2709., 4226., 2641., 5813.,  242., 1083., 6629., 4775., 5372.,
          224., 1452., 2910., 2753., 2644.,   49., 4897., 2621.,  565.,  850.,
          333., 1686., 1022., 3564.,  480., 1822., 4425., 1319., 1960., 6572.,
          659., 5203., 3679., 2977., 1037.,  182., 1672., 4241., 3089., 1488.,
         1082., 2837.,   93.,  322.,  248., 3425., 2530., 1253.,  461., 6633.,
          241., 6951., 4509., 2718.,  360.,  939.,  574.,  111., 2444.,  138.,
         3059., 4075., 2436.,  635., 43

In [13]:
gllis

[Graph(num_nodes=82, num_edges=494,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=210, num_edges=1442,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=86, num_edges=452,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=130, num_edges=668,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=157, num_edges=1058,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=121, num_edges=692,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)

In [5]:
gl

{'glabel': tensor([1632., 7044.,  343., 2259., 2115., 4744., 3320., 4082., 2656., 2821.,
          952., 1615., 5027.,  527., 4995.,  488., 6203.,  781., 3537., 4236.,
         3622., 2977., 4320., 4472., 3291.,  586.,  759., 6637., 4455.,  702.,
         1452., 3802., 3204., 5059., 1844., 6919., 2999., 4632.,  565., 5963.,
         4131., 5789., 1455., 2422., 6323., 3578., 3650., 5920., 2648.,  669.,
         1478., 1083., 5309.,  916., 3680., 1750., 5398., 3348., 3864.,  663.,
         6724., 6754., 4897., 3202.,  574., 5956., 6951., 4556., 6049., 5081.,
         1873., 3822., 5157.,  504., 1892., 7084., 4089.,  627., 5513., 3948.,
         2975.,  480., 1643., 4241., 5621., 5993.,  918.,  303., 4417., 1496.,
          740., 3250., 2426., 2040.,  532., 4383.,  688., 2837., 3962.,  699.,
         4736., 6465., 3980., 1213., 5291., 5831.,  635., 5982.,  560.,  999.,
         6815., 3823., 4589.,  248., 6461., 3059., 2054., 6824., 6156., 3534.,
          237., 2224., 4234., 3838.,  382.

In [6]:
gllis

[Graph(num_nodes=14, num_edges=62,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=157, num_edges=1058,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=251, num_edges=1574,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=91, num_edges=542,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=40, num_edges=212,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
       edata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}),
 Graph(num_nodes=28, num_edges=188,
       ndata_schemes={'w': Scheme(shape=(768,), dtype=torch.float32)}
 

In [9]:
nn= 0
g = load_graphs("./graph/graph_bin"+"/bidirec_"+"train"+"_"+"issue"+".bin")[0]
for gr in g:
    nn += gr.number_of_nodes()

In [4]:
print(nn) # issue

17059


In [6]:
print(nn) # fact

14427


In [8]:
print(nn) # fact ns

41441


In [10]:
print(nn) # issue ns

45371
